# Bias-Variance Tradeoff

## Learning Objectives
- Understand **bias** as the error from a model too simple to capture the true relationship
- Understand **variance** as the error from a model too sensitive to fluctuations in the training data
- Visualise the bias-variance tradeoff across polynomial degrees
- Identify underfitting and overfitting from train vs. generalisation error curves
- Know the informal rule: larger hypothesis class → lower bias, higher variance

## Problem Statement

Given training data $\{(x^{(i)}, y^{(i)})\}_{i=1}^{m}$ drawn iid from some distribution $\mathcal{D}$, we want to choose a model that **generalises** well to unseen data.

The **generalisation error** of a hypothesis $h$ is:
$$\varepsilon(h) = P_{(x,y) \sim \mathcal{D}}\bigl(h(x) \neq y\bigr)$$

Two failure modes exist:

| Failure mode | Cause | Symptom |
|---|---|---|
| **Underfitting** | Model too simple (high bias) | High train error AND high test error |
| **Overfitting** | Model too complex (high variance) | Low train error BUT high test error |

---

### Informal Definitions

- **Bias** — the expected generalisation error even if we could fit the model on an infinitely large training set; reflects whether the hypothesis class can represent the true function at all.
- **Variance** — how much the learned hypothesis varies when we retrain on different draws of the training set of size $m$; reflects sensitivity to noise in the finite sample.

The total generalisation error is roughly:
$$\text{Generalisation error} \approx \text{Bias}^2 + \text{Variance} + \text{Irreducible noise}$$

## 1. Visualising Underfitting, Good Fit, and Overfitting

Polynomial regression is the running example:
- **Degree 1** (linear): underfits — large bias, small variance
- **Degree 2–3** (quadratic/cubic): good fit — balanced bias and variance
- **Degree 5+** (high-degree): overfits — small bias, large variance

The figure below replicates the three-panel illustration from the notes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# True function: y = sin(2πx)
def true_fn(x):
    return np.sin(2 * np.pi * x)

m = 10
X_train = np.sort(np.random.uniform(0, 1, m))
y_train = true_fn(X_train) + np.random.normal(0, 0.2, m)

x_plot = np.linspace(0, 1, 300)
degrees = [1, 3, 9]
titles  = ['Degree 1 — Underfitting\n(High Bias, Low Variance)',
           'Degree 3 — Good Fit\n(Balanced)',
           'Degree 9 — Overfitting\n(Low Bias, High Variance)']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, deg, title in zip(axes, degrees, titles):
    coeffs = np.polyfit(X_train, y_train, deg)
    p      = np.poly1d(coeffs)

    ax.plot(x_plot, true_fn(x_plot), 'g-',  lw=2,   label='True function')
    ax.plot(x_plot, p(x_plot),       'b-',  lw=2.5, label=f'Degree-{deg} fit')
    ax.scatter(X_train, y_train, c='k', s=60, zorder=5, label='Training points')

    train_err = np.mean((p(X_train) - y_train) ** 2)
    gen_err   = np.mean((p(x_plot) - true_fn(x_plot)) ** 2)

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_ylim(-2.5, 2.5)
    ax.text(0.03, 0.97,
            f'Train MSE:  {train_err:.3f}\nApprox Gen MSE: {gen_err:.3f}',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))
    ax.legend(fontsize=8)

fig.suptitle('Underfitting vs. Overfitting (Polynomial Regression)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Bias-Variance Decomposition

For squared-loss regression the generalisation error decomposes exactly as:

$$\mathbb{E}\bigl[(h(x) - y)^2\bigr] = \underbrace{\bigl(\mathbb{E}[h(x)] - f(x)\bigr)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\bigl[(h(x) - \mathbb{E}[h(x)])^2\bigr]}_{\text{Variance}} + \sigma^2_{\text{noise}}
$$

where the expectations are over different draws of the training set.

We estimate this empirically by training many times on different samples of size $m$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def true_fn(x):
    return np.sin(2 * np.pi * x)

n_trials = 200
m        = 15
x_eval   = np.linspace(0, 1, 100)
degrees  = list(range(1, 10))

bias2_list = []
var_list   = []

for deg in degrees:
    preds = []
    for _ in range(n_trials):
        X = np.sort(np.random.uniform(0, 1, m))
        y = true_fn(X) + np.random.normal(0, 0.3, m)
        coeffs = np.polyfit(X, y, deg)
        preds.append(np.poly1d(coeffs)(x_eval))

    preds      = np.array(preds)           # (n_trials, 100)
    mean_pred  = preds.mean(axis=0)
    bias2      = np.mean((mean_pred - true_fn(x_eval)) ** 2)
    variance   = np.mean(preds.var(axis=0))

    bias2_list.append(bias2)
    var_list.append(variance)

bias2_arr = np.array(bias2_list)
var_arr   = np.array(var_list)
total_arr = bias2_arr + var_arr

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: bias² and variance curves
ax = axes[0]
ax.plot(degrees, bias2_arr, 'b-o', lw=2, ms=6, label='Bias²')
ax.plot(degrees, var_arr,   'r-s', lw=2, ms=6, label='Variance')
ax.plot(degrees, total_arr, 'k-^', lw=2, ms=6, label='Bias² + Variance')
ax.axvline(degrees[np.argmin(total_arr)], color='green', ls='--', lw=1.5,
           label=f'Optimal degree = {degrees[np.argmin(total_arr)]}')
ax.set_xlabel('Polynomial Degree'); ax.set_ylabel('Error')
ax.set_title('Bias-Variance Decomposition\n(averaged over 200 training sets)')
ax.legend(fontsize=9)

# Right: stacked area chart
ax = axes[1]
ax.stackplot(degrees, bias2_arr, var_arr,
             labels=['Bias²', 'Variance'], colors=['#4393c3', '#d6604d'], alpha=0.75)
ax.plot(degrees, total_arr, 'k-', lw=2.5, label='Total error')
ax.set_xlabel('Polynomial Degree'); ax.set_ylabel('Error')
ax.set_title('Bias² + Variance Stacked Area')
ax.legend(fontsize=9)

fig.suptitle('Empirical Bias-Variance Tradeoff', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Learning Curves

A **learning curve** plots train and test error as a function of training set size $m$.

| Model type | As $m$ increases… |
|---|---|
| High bias | Both errors converge to a high value; more data does not help |
| High variance | Test error decreases and approaches train error; more data helps |
| Well-chosen | Both errors converge to a low value |

This is the diagnostic tool for deciding whether to collect more data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

def true_fn(x):
    return np.sin(2 * np.pi * x)

m_values = list(range(5, 60, 3))
n_trials = 50
x_test   = np.linspace(0, 1, 200)
y_test   = true_fn(x_test)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
configs   = [(1, 'High Bias (Degree 1)'), (3, 'Good Fit (Degree 3)'), (9, 'High Variance (Degree 9)')]

for ax, (deg, title) in zip(axes, configs):
    train_errs, test_errs = [], []

    for m in m_values:
        tr_e, te_e = [], []
        for _ in range(n_trials):
            X = np.sort(np.random.uniform(0, 1, m))
            y = true_fn(X) + np.random.normal(0, 0.2, m)
            try:
                coeffs = np.polyfit(X, y, min(deg, m - 1))
                p      = np.poly1d(coeffs)
                tr_e.append(np.mean((p(X) - y) ** 2))
                te_e.append(np.mean((p(x_test) - y_test) ** 2))
            except np.linalg.LinAlgError:
                pass
        train_errs.append(np.mean(tr_e))
        test_errs.append(np.mean(te_e))

    ax.plot(m_values, train_errs, 'b-', lw=2, label='Train error')
    ax.plot(m_values, test_errs,  'r-', lw=2, label='Test error')
    ax.fill_between(m_values, train_errs, test_errs, alpha=0.15, color='purple',
                    label='Gap (variance)')
    ax.set_xlabel('Training set size $m$'); ax.set_ylabel('MSE')
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8.5)
    ax.set_ylim(0, min(1.5, max(test_errs) * 1.2))

fig.suptitle('Learning Curves — Diagnosing Bias vs. Variance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="464"
     viewBox="0 0 780 464" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Hypothesis class -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="40" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Hypothesis class &#x1d4d7;</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >controls model expressiveness &#x2014; size determines underfitting vs overfitting</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="78" font-size="11.5" font-style="italic" fill="#555"
        >ERM: &#x125; = argmin &#x3b5;&#x302;(h) over training set</text>
  <text x="114" y="96" font-size="11.5" font-style="italic" fill="#555"
        >of size m drawn i.i.d. from &#x1d49f;</text>

  <!-- Row 2: Learned hypothesis -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Learned hypothesis &#x125;</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >varies across different training sets &#x2014; average behaviour vs spread</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="178" font-size="11.5" font-style="italic" fill="#555"
        >decompose generalisation error</text>
  <text x="114" y="196" font-size="11.5" font-style="italic" fill="#555"
        >over draws of training set</text>

  <!-- Row 3: Bias&#xb2; + Variance -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Bias&#xb2; + Variance</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >Bias&#xb2; &#x2193; with complexity, Variance &#x2191; with complexity &#x2014; U-shaped total error</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >find minimum of Bias&#xb2; + Variance</text>

  <!-- Row 4: Optimal complexity -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Optimal model</text>
  <text x="102" y="352" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">complexity</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >diagnosed via learning curves &#x2014; gap between train and test error</text>

  <!-- step 4&#x2192;5 -->
  <line x1="102" y1="358" x2="102" y2="408"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 5: Model selection -->
  <rect x="10" y="412" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="440" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Model selection</text>
  <line x1="197" y1="435" x2="216" y2="435"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="412" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >choose complexity via cross-validation or regularisation (covered later)</text>
</svg>
""")

## Summary

| Concept | Definition | Key Insight |
|---|---|---|
| Bias | Expected error even with infinite training data | Reflects whether $\mathcal{H}$ can represent the true function |
| Variance | Variability of $\hat{h}$ across different training sets of size $m$ | Reflects sensitivity to finite training sample |
| Underfitting | High bias — high train error **and** high test error | Hypothesis class too small |
| Overfitting | High variance — low train error, high test error | Hypothesis class too large |
| Generalisation error | $\approx \text{Bias}^2 + \text{Variance} + \sigma^2_{\text{noise}}$ | Total error is U-shaped in model complexity |
| Learning curve | Train/test error plotted vs. training set size $m$ | High bias: both errors converge high; high variance: gap narrows as $m$ grows |

**Key insight:** model complexity cannot be tuned to simultaneously eliminate both bias and variance — the sweet spot balances the two and is found via cross-validation or regularisation.